In [1]:
import json
import chromadb
from chromadb.utils import embedding_functions
import pandas as pd
import pytrec_eval
from tqdm import tqdm
import numpy as np
import ast
import os
#d = ast.literal_eval(l)

In [2]:
##load propositions
path = '/scratch/NLU/cvlachos/SCQA/Samu_XLSR_finetuning/data/propositions.json'

with open(path) as f:
    d = json.load(f)


propositions = d[0]['Propositions']
prop_dict = {i:f'id_{n}' for n, i in enumerate(propositions)}


In [ ]:
# # ##load dataframe
# dataframe_path = '/scratch/NLU/cvlachos/SCQA/Samu_XLSR_finetuning/data/split0_500/step4_fixed_props_test_text_implicit.parquet'
# df = pd.read_parquet(dataframe_path)
# df['propositions_ids'] = df.propositions.apply(lambda x : [prop_dict[i] for i in x])


In [3]:
splits = [i for i in os.listdir('/scratch/NLU/cvlachos/SCQA/Samu_XLSR_finetuning/data/') if 'split' in i]
df = pd.DataFrame()
for i in splits:
    #dataframe_path = os.path.join('/scratch/NLU/cvlachos/SCQA/Samu_XLSR_finetuning/data/',i,'step4_fixed_props_test_text_implicit.parquet')
    dataframe_path = os.path.join('/scratch/NLU/cvlachos/SCQA/Samu_XLSR_finetuning/data/',i,'step4_fixed_props_test_text_implicit_original_dataset.parquet')

    df = pd.concat([df, pd.read_parquet(dataframe_path)])
df['propositions_ids'] = df.propositions.apply(lambda x : [prop_dict[i] for i in x])

df.shape


(5285, 10)

In [4]:
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/LaBSE"
)

client = chromadb.PersistentClient(path="/scratch/NLU/cvlachos/SCQA/Samu_XLSR_finetuning/data/chroma_db")
collection = client.get_or_create_collection(
    name="propositions_VS",
    embedding_function=sentence_transformer_ef
)

In [5]:
eval_dict = {
    'map': 0.,
    'recip_rank': 0.,
    'recall_5': 0.,
    'recall_10': 0.,
    'recall_20': 0.,
    'recall_50': 0.,
    'recall_100': 0.,
  }

for row in tqdm(range(0,df.shape[0])):
    #print(df.iloc[row].rewrite)
    query_embedding = [np.array(df.iloc[row].Embeddings)]
    ground_truth_props={'q':{k:1 for k in df.iloc[row].propositions_ids}}
    #print(ground_truth_props)
    query_embedding = query_embedding[0].squeeze()

    #query_embedding = sentence_transformer_ef([df.iloc[0].rewrite])

    #print(query_embedding.shape)
    

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=100
    )
    predictions = {'q':{k:1/(n+1) for n,k in enumerate(results['ids'][0])}}
    #print(predictions)

      
    evaluator = pytrec_eval.RelevanceEvaluator(
      ground_truth_props, {"map", "recip_rank", "recall_5", "recall_10", "recall_20", "recall_50", "recall_100"})

    results = evaluator.evaluate(predictions)
    #print(results)
    
    for i in results:
      for j in results[i]:
        eval_dict[j] += results[i][j]/df.shape[0]
  

100%|██████████| 5285/5285 [01:07<00:00, 78.82it/s]


All

In [ ]:
eval_dict #ASR

{'map': 0.3236759219229171,
 'recip_rank': 0.3970571673829459,
 'recall_5': 0.4117195618425498,
 'recall_10': 0.49368974082786354,
 'recall_20': 0.5680163623825027,
 'recall_50': 0.6593012170212038,
 'recall_100': 0.7249210681046415}

In [6]:
eval_dict #Original

{'map': 0.3502822131860492,
 'recip_rank': 0.42781247735434846,
 'recall_5': 0.44034551383652404,
 'recall_10': 0.5264174981961177,
 'recall_20': 0.5987082727007201,
 'recall_50': 0.6921603640761943,
 'recall_100': 0.7512308662640196}